# SNA Topic C.10: BA-Shapes - Proposed Enhanced GNN Explainability

> **Môn học:** Phân Tích Mạng Xã Hội (SNA) - UIT 2026  
> **Giảng viên hướng dẫn:** TS. Trần Hùng Nghiệp (`nghiepth@uit.edu.vn`)  
> **Đề tài:** Topic C.10 - Making Graph Machine Learning Models Explainable  
> **Định hướng nghiên cứu:** Đánh giá các thuật toán giải thích (XAI) trên mô hình GNN Đề xuất Cải tiến (Proposed GCN 3 lớp, mã hóa vị trí bước ngẫu nhiên RWPE 4 chiều) trên đồ thị BA-Shapes. Đo lường Fidelity+ dương toàn diện và Motif-ROC-AUC đối chiếu Ground-Truth House motif.  

---
### Hướng dẫn thực thi 1-Click trên Google Colab:
1. Chọn Runtime GPU: **Runtime -> Change runtime type -> T4 GPU** (hoặc CPU).
2. Bấm **Runtime -> Run all** để thực thi toàn bộ quy trình tự động.
3. Kết quả (JSON, CSV, logs) được tự động đóng gói và tải về máy tính.

## Bước 1: Kết nối Google Drive & Kiểm tra Môi trường GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')


## Bước 2: Cài đặt Thư viện Học thuật (PyG, Captum, SciPy, OGB)

In [ ]:
# Cài đặt an toàn: Giữ nguyên NumPy 2.x & Torch 2.11 mặc định của Colab Python 3.13
# Cài các thư viện GNN phụ thuộc qua pure wheels, không ép compile C++
!pip install -q torch-geometric captum ogb tabulate
print('Đã cài đặt thành công tất cả các thư viện phụ thuộc (Colab Python 3.13 Ready)!')


## Bước 3: Thiết lập Thư mục Làm việc & Đồng bộ Mã nguồn

In [ ]:
import os
import shutil
import zipfile

WORK_DIR = '/content/mxh_project'
os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)

# Cách 1: Tự động nhận diện nếu đã kéo thả mxh_bundle.zip vào Colab
if os.path.exists('/content/mxh_bundle.zip'):
    print('Đang giải nén mxh_bundle.zip vào thư mục làm việc...')
    with zipfile.ZipFile('/content/mxh_bundle.zip', 'r') as z:
        z.extractall(WORK_DIR)
    print('>>> ĐỒNG BỘ THÀNH CÔNG TOÀN BỘ MÃ NGUỒN VÀ DỮ LIỆU (5.37 MB)!')

# Cách 2: Đồng bộ từ Google Drive nếu có thư mục MXH
elif os.path.exists('/content/drive/MyDrive/MXH'):
    print('Đồng bộ mã nguồn từ Google Drive...')
    !rsync -av --exclude='.venv' /content/drive/MyDrive/MXH/ '{WORK_DIR}/'
    print('>>> ĐỒNG BỘ THÀNH CÔNG TỪ GOOGLE DRIVE!')

# Cách 3: Nút bấm trực tiếp để chọn file mxh_bundle.zip từ máy tính
else:
    print('Chưa thấy mã nguồn. Bạn hãy bấm nút "Choose Files" bên dưới để tải file mxh_bundle.zip (5.37 MB) từ máy lên:')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith('.zip'):
            with zipfile.ZipFile(f'/content/{fname}', 'r') as z:
                z.extractall(WORK_DIR)
            print(f'>>> ĐÃ GIẢI NÉN THÀNH CÔNG {fname} VÀO {WORK_DIR}!')


## Bước 4: Thực thi Đánh giá Giải thích XAI-GNN
*SNA Topic C.10: BA-Shapes - Proposed Enhanced GNN Explainability*

In [ ]:
!python -m src.pipeline.run_scenarios --scenario proposed --dataset bashapes --nodes 100 --budget-k 5


## Bước 5: Đóng gói & Tự động Tải Kết quả Về Máy

In [ ]:
import zipfile
from google.colab import files

zip_name = '/content/SNA_C10_BAShapes_Proposed_Explainability_Results.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, _, fs in os.walk('experiments/logs'):
        for f in fs:
            fp = os.path.join(root, f)
            z.write(fp, os.path.relpath(fp, '.'))

DRIVE_OUT = '/content/drive/MyDrive/SNA_C10_Results'
os.makedirs(DRIVE_OUT, exist_ok=True)
shutil.copy(zip_name, os.path.join(DRIVE_OUT, 'SNA_C10_BAShapes_Proposed_Explainability_Results.zip'))
print('Đã lưu thành công vào Drive: ' + os.path.join(DRIVE_OUT, 'SNA_C10_BAShapes_Proposed_Explainability_Results.zip'))
files.download(zip_name)
